Mit Colab kann **dieses Jupyter-Notebook interaktiv im Browser** gestartet werden:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DonGoginho/myPy/blob/main/parlamentsdienste_ris_api/pd_ris_api_al_geschaefte.ipynb)

Und hier ein Jupyter-Notebook zur [generellen Verwendung der RIS-API](https://colab.research.google.com/github/opendatazurich/opendatazurich.github.io/blob/master/ris-api/RIS-API-Beispiele.ipynb)
# Python-Abfrage an RIS-API nach Geschäftern, die von der Alternativen Liste eingegeben wurden

Das Datenmodell der [**RIS-API ist als Swagger File**](https://opendatazurich.github.io/ris-api/docs/) verfügbar. Für das Verständnis ist dieses daher wichtig. 

**Inhaltsverzeichnis:**

1. [Mitglieder der Alternativen Liste](#Mitglieder-der-AL)
1. [Gemeinderats-Geschäfte](#Gemeinderats-Geschäfte)

In [1]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from pprint import pprint
import pandas as pd
import random

In [3]:
SSL_VERIFY = False
# evtl. SSL_VERIFY auf False setzen wenn die Verbindung zu https://www.gemeinderat-zuerich.ch nicht klappt (z.B. wegen Proxy)
# Um die SSL Verifikation auszustellen, bitte die nächste Zeile einkommentieren ("#" entfernen)
# SSL_VERIFY = False

In [4]:
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings()

## Infos zu GLP
- 'parteiId': '0a8eb90f-6d09-47ac-99fe-ab97a77d4954'
- 'fraktionsId': '8a269148-2a01-45f9-9e1d-c2b11d07cd31'

## Mitglieder der GLP

Um alle Mitglieder einer Partei zu bekommen, müssen folgende Abfragen gemacht werden:
    
    1. ID der Partei holen
    1. Suche nach Mitgliedern der Partei (mit dieser ID)
    1. *Optional:* Detail-Informationen eines Mitglieds holen
    
    

### Alle relevanten Mitglieder-Parameter holen

Lese alle Parameter zu den Mitgliedern in `params` ein

In [5]:
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Mitglieder/parameter', headers=headers, verify=SSL_VERIFY)
params = r.json()
#params

Lese die Informationen zu den Parteien aus `params`heraus

In [6]:
# get id of AL
parties = params['Parteien']
#party = {'parteiId': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8'}
parties

[{'Name': 'AL', 'Id': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8'},
 {'Name': 'CVP', 'Id': '5964b653-a32a-4e0c-92a1-07720566a92c'},
 {'Name': 'EVP', 'Id': '187b7ad1-2095-43cc-b257-e9834a54a961'},
 {'Name': 'FDP', 'Id': '8240033e-4818-4406-8f73-b7829d99ddee'},
 {'Name': 'GLP', 'Id': '0a8eb90f-6d09-47ac-99fe-ab97a77d4954'},
 {'Name': 'Grüne', 'Id': 'fe956d66-e14d-4739-82cf-c12202d96445'},
 {'Name': 'Parteilos', 'Id': '9e037de7-40a4-4fa3-85b7-f4c30cbb26c8'},
 {'Name': 'SP', 'Id': '6cb1a3bb-8e26-4818-a8f5-0ea49ab3cd57'},
 {'Name': 'SVP', 'Id': '8e484ad9-6e46-4d39-bc05-fbf18be11b0c'}]

Für diese Anfrage ist die AL relevant. Definiere daher als `party`die gesuchte Partei

In [7]:
party = {'parteiId': '0a8eb90f-6d09-47ac-99fe-ab97a77d4954','Name': 'GLP'}
party

{'parteiId': '0a8eb90f-6d09-47ac-99fe-ab97a77d4954', 'Name': 'GLP'}

### Mitglieder der AL abfragen

In der Query wird die ParteiId angegeben, welche für die Abfrage des Endpoints relevant ist.
In `members`wird das Resultat der Abfrage als JSON-Array eingelesen.

In [8]:
query = {'parteiId':  '0a8eb90f-6d09-47ac-99fe-ab97a77d4954'}
#query = {'fraktionsId': '8a269148-2a01-45f9-9e1d-c2b11d07cd31'}
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Mitglieder/suchen', params=query, headers=headers, verify=SSL_VERIFY)
members = r.json()
#members

### Details zu den AL-Mitgliedern abfragen

Mit dem `/Mitglieder/details` Endpunkt können Details eines Mitglieds abgefragt werden. Dazu muss die Mitglieder-ID jedes Mitglieds herausgelesen und über die API abgefragt werden. Dies erfolgt über die Funktion `get_member_details()`.

In [9]:
#members

In [10]:
def get_member_details(member):
    #print(member)
    query = {'mid': member['Id']}
    #print(query)
    headers = {'Accept': 'application/json'}
    r = requests.get('https://www.gemeinderat-zuerich.ch/api/Mitglieder/details', params=query, headers=headers, verify=SSL_VERIFY)
    details = r.json()
    
    # trim "Wahlkreis" and "Wohnkreis" since they have a padding
    details['Wahlkreis'] = details['Wahlkreis'].strip()
    if details['Wohnkreis']:
        details['Wohnkreis'] = details['Wohnkreis'].strip()
    # add original member to keep WahlkreisOrderBy column
    details.update(member)
    return details
    
member_details = [get_member_details(m) for m in members]
df = pd.DataFrame(member_details)
df

,Id,Name,Vorname,Anrede,Titel,Geburtstag,Wohnkreis,Beruf,Partei,Fraktion,...,EmailPrivat,EmailGeschaeftlich,Mobiltelefon,MobiltelefonGeschaeftlich,TelefonGeschaeftlich,TelefonPrivat,Internetauftritt,Interessenverbindungen,NameInUrl,WahlkreisOrderBy
0,32e89ef8-f5b2-46e5-aafe-7dd15929f00a,Baumann,Markus,Herr,None,1973-09-19T00:00:00,None,Fachmann für Arbeitsintegration/Job Coach,GLP,GLP,...,baumann-markus@bluewin.ch,,076 396 19 20,None,None,None,,None,Markus%20Baumann,105
1,e19095bc-beed-43e2-9e39-685c46f8d577,Cavalli,Nicolas,Herr,None,1982-11-09T00:00:00,4,Consultant,GLP,GLP,...,nicolas.cavalli@grunliberale.ch,None,078 602 27 70,None,None,None,None,None,Nicolas%20Cavalli,102
2,6a6e0c52-3ae0-48d3-b7d3-a9412dad103b,Garcia,Isabel,Frau,None,1963-06-15T00:00:00,3,Politologin & Kommunikationsberaterin,GLP,GLP,...,garciaisabel_@hotmail.com,,079 652 85 19,None,,,,"- Secondas Zürich, Präsidentin\r- Mitglied des...",Isabel%20Garcia,101
3,a0692db6-12d8-462b-a02d-b3b3f3045551,Hofer Frei,Simone,Frau,None,1974-12-16T00:00:00,6,Redaktorin / Ökonomin (lic. rer. pol.),GLP,GLP,...,simone.hofer@econtext.ch,simone.hofer@econtext.ch,078 612 08 21,078 612 08 21,None,,www.econtext.ch,"- LAC TV Unterstrass, Zürich, Trainerin\r- Vét...",Simone%20Hofer%20Frei,103
4,aa77152d-5fde-4725-8c69-21c022000ec8,Krayenbühl,Guy,Herr,,1968-01-29T00:00:00,3,Staatsanwalt,GLP,GLP,...,mail@guykrayenbuehl.ch,,076 386 86 16,,044 248 21 11,None,None,None,Guy%20Krayenb%c3%bchl,100
5,6f80de0e-2322-46c4-891b-3a2781b127c9,Landolt,Maleica,Frau,None,1967-06-05T00:00:00,11,Pflegefachfrau HF,GLP,GLP,...,maleika.landolt@vtxmail.ch,None,079 317 33 71,None,044 811 07 77,043 300 12 60,None,"- Rotkreuzdienst Schweiz, Bern, OGfr. RKD+\r- ...",Maleica%20Landolt,107
6,09fccfe6-c6e3-487e-8efc-4db1e8af3d8f,Merki,Markus,Herr,None,1977-10-04T00:00:00,11,Bau- und Wirtschaftsingenieur FH,GLP,GLP,...,markus.merki@hotmail.com,,,None,,None,None,"- Grünliberale Kreis 11 & 12, Vorstandsmitglie...",Markus%20Merki,107
7,f5ae2bc7-e3c4-4f2e-96b8-0b4c1abbb87d,Monn,Christian,Herr,Dr.,1959-12-13T00:00:00,11,wissenschaftlicher Mitarbeiter / Dozent,GLP,GLP,...,cmonn@hispeed.ch,,,None,,,www.christianmonn.ch,None,Christian%20Monn,108
8,162e20b7-31de-46ef-b520-0021c3e269cb,Nabholz,Ann-Catherine,Frau,Dr.,1968-04-18T00:00:00,7,Dozentin,GLP,GLP,...,ann-catherine.nabholz@grunliberale.ch,None,079 561 49 82,None,None,043 488 69 85,None,"- Theater Neumarkt AG, Mitglied Verwaltungsrat",Ann-Catherine%20Nabholz,104
9,1a2aa558-cbd2-4ab6-8725-7c791cec92ad,Novak,Martina,Frau,None,1985-01-16T00:00:00,8,Leiterin Politik,GLP,GLP,...,martinanovak.politik@gmail.com,,,None,,None,None,None,Martina%20Novak,104


In [11]:
# Beispiel einer Grafik dazu
#count_by_wahlkreis = df[['Wahlkreis', 'WahlkreisOrderBy']].groupby(['Wahlkreis', 'WahlkreisOrderBy']).size().reset_index(name='Anzahl')
#count_by_wahlkreis = count_by_wahlkreis.sort_values(by=['WahlkreisOrderBy'])
#count_by_wahlkreis.plot.bar(x='Wahlkreis', y='Anzahl')

# Anfrage der Parlamentsdienste

Das Interesse gilt allen durch ein GLP-Ratsmitglied oder die GLP-Fraktion eingereichten Vorstösse in dieser Legislatur.  

Es müsste im Datensatz in etwa folgender Filter gesetzt werden:
- Geschäftsart=Motion, Postulat, Interpellation, Schriftliche Anfrage, Beschlussantrag oder Globalbudgetantrag
- Akteure/Einreichende Person=GLP (muss der Partei "GLP" angehören)
- Zeitraum=16.05.2018 bis 03.05.2022 

IDs zur GLP
- 'parteiId':  '0a8eb90f-6d09-47ac-99fe-ab97a77d4954'
- 'fraktionsId': '8a269148-2a01-45f9-9e1d-c2b11d07cd31'

Resultat: 
Liste mit Angaben zu
- Geschäftsnummer
- Geschäftstitel
- Geschäftsart
- Einreichende Person/Partei/Fraktion: GLP
- Link auf das Geschäft auf der Website des Gemeinderats



## Gemeinderats-Geschäfte die von der GLP eingereicht wurden

Im der Resultatsliste sollen alls Geschäfte aufgelistet werden, welche die GLP entweder als Partei oder als Fraktion eingegeben hat.

1.   Stelle zuerst den Request auf die Geschäfte, welche von der GLP **als Fraktion** eingegeben wurden und übergebe das Resultat aus einem JSON (`result_geschaefte_fraktion`) in einen Dataframe (`df_deals_fraktion`).

In [12]:
#query = {'personId': member['Id']}

# via swagger: URL mit einem OR Operatoren drin
#https://www.gemeinderat-zuerich.ch/api/Geschaeft/suchen?parteiId=df5ccd54-009e-40ae-b687-4ce88a71f5d8|fraktionId=53bf7290-4055-455b-bb1f-2ba64e76294d&orderBy=geschaeftsart

query_fraktion = {'fraktionId': '8a269148-2a01-45f9-9e1d-c2b11d07cd31', 'activePage':1, 'pageSize':700}
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Geschaeft', 
                 params=query_fraktion, 
                 headers=headers, 
                 verify=SSL_VERIFY)
result_geschaefte_fraktion = r.json()
df_deals_fraktion = pd.DataFrame(result_geschaefte_fraktion['Geschaefte'])
print('done')




done


In [13]:
df_deals_fraktion.info

<bound method DataFrame.info of                               GeschaeftId  Geschaeftsjahr  Geschaeftsnummer  \
0    b21dde90-7652-44e6-bc1f-abf23b760d12            2022               103   
1    25e80ba4-9deb-4f85-8d0a-f35fa1160498            2022                92   
2    5c2bf33e-5ff8-48f7-a7b2-1b3b783dc0d0            2022                88   
3    e31012f6-347b-48cd-a7cb-fbfbca017713            2022                76   
4    41dedcfb-0e34-4456-bddd-47eb70a78dbd            2022                69   
..                                    ...             ...               ...   
235  0502e52c-9ce0-466f-9503-826c6cdb3799            2010               518   
236  15ef01f9-7967-459c-8063-a18785272bb2            2010               421   
237  168944e8-5535-4bfa-894d-537f7cca085c            2010               323   
238  7c62d067-f93a-4764-9623-4cda8fa09f19            2010               307   
239  06bfef68-7dbe-46c6-a84a-c0de5bfccf57            2010               239   

                   

2.   Stelle anschliessend den Request auf die Geschäfte, welche von der GLP **als Partei** eingegeben wurden und übergebe das Resultat aus einem JSON (result_geschaefte_partei) in einen Dataframe (df_deals_partei).

In [14]:
query_partei = {'parteiId': '0a8eb90f-6d09-47ac-99fe-ab97a77d4954', 'activePage':1, 'pageSize':700}
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Geschaeft', 
                 params=query_partei, 
                 headers=headers, 
                 verify=SSL_VERIFY)
result_geschaefte_partei = r.json()
df_deals_partei = pd.DataFrame(result_geschaefte_partei['Geschaefte'])
print('done')


done


In [15]:
df_deals_partei.info

<bound method DataFrame.info of                               GeschaeftId  Geschaeftsjahr  Geschaeftsnummer  \
0    61720d59-0a1a-45b5-820a-2f0f988f0fdf            2022               133   
1    72a578b2-d377-4b35-9cbf-e72ce850317f            2022               108   
2    bc475206-69c9-4b10-9ebb-55ecddb1c75f            2022               107   
3    5b38e1d4-8c90-4502-8e2c-a383e314c6cd            2022               106   
4    54de7aef-848d-4f75-b2d1-0618bdcb5365            2022                93   
..                                    ...             ...               ...   
331  f1e133b0-9364-43b4-8aac-05108476ed32            2005               291   
332  f174da4d-ee17-49d1-9732-3aa9b47e6050            2005               210   
333  ddb76021-d2da-4d09-8907-faff44891e0f            2004               611   
334  9b3b28bd-bb24-46f3-bb15-4329aa1d2207            2004               533   
335  b2e323e3-5ed8-4a12-a4c4-5a6ad91a01d6            2004               150   

                   

Füge nun beide DataFrames der Teilanfragen (1 und 2) zusammen. 
`ignore_index` ist hier wichtig, sonst funktioniert das zusammenhängen nicht wie erwünscht.
Sortiere danach den DataFrame absteigend nach Geschäftsjahr und Geschäftsnummer und schreibe ihn in den DataFrame df_deals.


In [16]:
#df_deals_partei.columns
df_deals_partei.dtypes
#df_deals_fraktion.dtypes

GeschaeftId         object
Geschaeftsjahr       int64
Geschaeftsnummer     int64
Geschaeftstitel     object
Geschaeftsart       object
dtype: object

In [17]:
df_deals_appended = df_deals_partei.append(df_deals_fraktion, verify_integrity=True, ignore_index=True, sort=True)
df_deals = df_deals_appended.sort_values(by=['Geschaeftsjahr', 'Geschaeftsnummer'], ascending=False)
df_deals
#result = df1.append(df4, sort=False)

,GeschaeftId,Geschaeftsart,Geschaeftsjahr,Geschaeftsnummer,Geschaeftstitel
0,61720d59-0a1a-45b5-820a-2f0f988f0fdf,Schriftliche Anfrage,2022,133,"Fehlende Krankentaggeldversicherung, Gründe fü..."
1,72a578b2-d377-4b35-9cbf-e72ce850317f,Postulat,2022,108,Zeitlich begrenzte berufliche und schulische A...
2,bc475206-69c9-4b10-9ebb-55ecddb1c75f,Postulat,2022,107,Einrichtung einer zeitlich begrenzten Anlaufst...
3,5b38e1d4-8c90-4502-8e2c-a383e314c6cd,Postulat,2022,106,Errichtung einer Jobplattform für Personen mit...
336,b21dde90-7652-44e6-bc1f-abf23b760d12,Beschlussantrag,2022,103,Rückkehr des Gemeinderats in das Rathaus
...,...,...,...,...,...
331,f1e133b0-9364-43b4-8aac-05108476ed32,Postulat,2005,291,"Geschäftsberichte, Bezugnahme auf Rechnungsabs..."
332,f174da4d-ee17-49d1-9732-3aa9b47e6050,Schriftliche Anfrage,2005,210,"Seebad Utoquai, vorzeitige Schliessung am 15.8..."
333,ddb76021-d2da-4d09-8907-faff44891e0f,Interpellation,2004,611,"Villa Mooser, Mietkonditionen für den künftige..."
334,9b3b28bd-bb24-46f3-bb15-4329aa1d2207,Postulat,2004,533,"Bewilligungsverfahren von Amtsstellen, Vereinf..."


In [18]:
df_deals = df_deals_appended.sort_values(by=['Geschaeftsjahr', 'Geschaeftsnummer'], ascending=False)


Check Output des Dataframes

In [19]:
df_deals.head()

,GeschaeftId,Geschaeftsart,Geschaeftsjahr,Geschaeftsnummer,Geschaeftstitel
0,61720d59-0a1a-45b5-820a-2f0f988f0fdf,Schriftliche Anfrage,2022,133,"Fehlende Krankentaggeldversicherung, Gründe fü..."
1,72a578b2-d377-4b35-9cbf-e72ce850317f,Postulat,2022,108,Zeitlich begrenzte berufliche und schulische A...
2,bc475206-69c9-4b10-9ebb-55ecddb1c75f,Postulat,2022,107,Einrichtung einer zeitlich begrenzten Anlaufst...
3,5b38e1d4-8c90-4502-8e2c-a383e314c6cd,Postulat,2022,106,Errichtung einer Jobplattform für Personen mit...
336,b21dde90-7652-44e6-bc1f-abf23b760d12,Beschlussantrag,2022,103,Rückkehr des Gemeinderats in das Rathaus


Exportiere die Geschäfte dieser Anfrage als Excel und als CSV.


In [20]:
#df_deals.to_csv('deals.csv')  
df_deals.to_excel('glp_deals.xlsx')  

## Details der Gemeinderats-Geschäfte abfragen

Die Abfrage über den Geschäftsendpoint ('https://www.gemeinderat-zuerich.ch/api/Geschaeft') liefert keine detaillierten Daten. Z.B wer das Geschäft eingereicht hat oder wie der aktuelle Stand dabei ist.

Dazu muss der Endpoint zu den Geschäftsdetails ('https://www.gemeinderat-zuerich.ch/api/Geschaeft/details') abgefragt werden. Dieser benötigt als Input die ID des Geschäfts (die gid).


In [21]:
def get_geschaefte_details(index):
    #Herauslesen der GeschaeftId aus dem Dataframe df_deals
    query = {'gid': df_deals['GeschaeftId'][index]}
    #print(query)
    headers = {'Accept': 'application/json'}
    r = requests.get('https://www.gemeinderat-zuerich.ch/api/Geschaeft/details', params=query, headers=headers, verify=SSL_VERIFY) 
    details = r.json()
    # Da komischerweise in den Details die gid nicht mehr vorkommt, weise ich sie hier explizit zu, damit sie im Output vorkommt.
    details['gid'] = df_deals['GeschaeftId'][index]
    return details
 
#Iterrows ist nicht die schnellste Variante der Abfrage. Aber hier dennoch zweckmässig.
deals_details = [get_geschaefte_details(index) for index, row in df_deals.iterrows()]
print("done")




done


Einlsesen des JSON-Outputs in einen DataFrame.
Beachte die Struktur des Outputs. Siehe auch https://opendatazurich.github.io/ris-api/docs/#/Geschaeft/get_Geschaeft_details > Model

In [22]:
df_deals_details = pd.DataFrame(deals_details)
df_deals_details



,GeschaeftTitel,GeschaeftsJahr,GeschaeftsNummer,GeschaeftsArt,Erstunterzeichnender,ZweiterErstunterzeichnender,EinreichendeGruppe,PendentBei,FristBis,Departemente,Ablaufschritte,VerknuepfteGeschaefte,gid
0,"Fehlende Krankentaggeldversicherung, Gründe fü...",2022,133,Schriftliche Anfrage,"{'Id': 'e19095bc-beed-43e2-9e39-685c46f8d577',...","{'Id': 'e77ac228-d6c5-42ee-9862-6a2cfeb1605f',...",[],Stadtrat,2022-07-06T21:26:12.1,"[{'Id': 1, 'Name': 'Zuteilung ausstehend'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': '8d6ea382-eeb3-45a8-a7dd-8886...,61720d59-0a1a-45b5-820a-2f0f988f0fdf
1,Zeitlich begrenzte berufliche und schulische A...,2022,108,Postulat,"{'Id': '32e89ef8-f5b2-46e5-aafe-7dd15929f00a',...","{'Id': '6a6e0c52-3ae0-48d3-b7d3-a9412dad103b',...",[],Gemeinderat,None,"[{'Id': 11, 'Name': 'Sozialdepartement (SD)'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': 'bc475206-69c9-4b10-9ebb-55ec...,72a578b2-d377-4b35-9cbf-e72ce850317f
2,Einrichtung einer zeitlich begrenzten Anlaufst...,2022,107,Postulat,"{'Id': '32e89ef8-f5b2-46e5-aafe-7dd15929f00a',...","{'Id': '6a6e0c52-3ae0-48d3-b7d3-a9412dad103b',...",[],Gemeinderat,None,"[{'Id': 11, 'Name': 'Sozialdepartement (SD)'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': '72a578b2-d377-4b35-9cbf-e72c...,bc475206-69c9-4b10-9ebb-55ecddb1c75f
3,Errichtung einer Jobplattform für Personen mit...,2022,106,Postulat,"{'Id': 'aa77152d-5fde-4725-8c69-21c022000ec8',...","{'Id': 'e77ac228-d6c5-42ee-9862-6a2cfeb1605f',...",[],Gemeinderat,None,"[{'Id': 11, 'Name': 'Sozialdepartement (SD)'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': '72a578b2-d377-4b35-9cbf-e72c...,5b38e1d4-8c90-4502-8e2c-a383e314c6cd
4,Rückkehr des Gemeinderats in das Rathaus,2022,103,Beschlussantrag,None,None,"[{'Name': 'AL', 'Id': '53bf7290-4055-455b-bb1f...",,0001-01-01T00:00:00,[],"[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[],b21dde90-7652-44e6-bc1f-abf23b760d12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,"Geschäftsberichte, Bezugnahme auf Rechnungsabs...",2005,291,Postulat,"{'Id': 'bf3cdcb5-e2dd-44e0-8cf3-f21d1ccb6e0e',...","{'Id': 'fdfcd10d-7734-47af-b898-83e478aa07b9',...",[],,0001-01-01T00:00:00,"[{'Id': 4, 'Name': 'Finanzdepartement (FD)'}]","[{'Datum': '2008-09-17T00:00:00', 'Created': '...",[],f1e133b0-9364-43b4-8aac-05108476ed32
572,"Seebad Utoquai, vorzeitige Schliessung am 15.8...",2005,210,Schriftliche Anfrage,"{'Id': '87cf2c8f-edda-4afa-b680-a683e1601fba',...","{'Id': 'fdfcd10d-7734-47af-b898-83e478aa07b9',...",[],,0001-01-01T00:00:00,"[{'Id': 7, 'Name': 'Tiefbau- und Entsorgungsde...","[{'Datum': '2005-10-26T00:00:00', 'Created': '...",[],f174da4d-ee17-49d1-9732-3aa9b47e6050
573,"Villa Mooser, Mietkonditionen für den künftige...",2004,611,Interpellation,"{'Id': 'fdfcd10d-7734-47af-b898-83e478aa07b9',...",None,[],,0001-01-01T00:00:00,"[{'Id': 4, 'Name': 'Finanzdepartement (FD)'}]","[{'Datum': '2007-05-09T00:00:00', 'Created': '...",[],ddb76021-d2da-4d09-8907-faff44891e0f
574,"Bewilligungsverfahren von Amtsstellen, Vereinf...",2004,533,Postulat,"{'Id': 'fdfcd10d-7734-47af-b898-83e478aa07b9',...",None,[],,0001-01-01T00:00:00,"[{'Id': 3, 'Name': 'Präsidialdepartement (PRD)'}]","[{'Datum': '2007-09-19T00:00:00', 'Created': '...",[],9b3b28bd-bb24-46f3-bb15-4329aa1d2207


In der Anfrage ist ein Link auf das Geschäft gewünscht. Die URL hat einen fixen Teil an den am Ende die gid des Geschäfts angehängt wird.
Da der Link nicht im Datenmodell vorhanden ist, wird er hier erstellt und dem DataFrame angehängt.


In [23]:
#df_deals_details['joined_col'] = df_deals_details.GeschaeftsArt.str.cat(df_deals_details.PendentBei)
basis_url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft?gid="

df_deals_details['deals_details_url'] = basis_url_geschaefte + df_deals_details.gid

df_deals_details.head(3)
#print(basis_url_geschaefte +''+ df_deals_details.gid)


,GeschaeftTitel,GeschaeftsJahr,GeschaeftsNummer,GeschaeftsArt,Erstunterzeichnender,ZweiterErstunterzeichnender,EinreichendeGruppe,PendentBei,FristBis,Departemente,Ablaufschritte,VerknuepfteGeschaefte,gid,deals_details_url
0,"Fehlende Krankentaggeldversicherung, Gründe fü...",2022,133,Schriftliche Anfrage,"{'Id': 'e19095bc-beed-43e2-9e39-685c46f8d577',...","{'Id': 'e77ac228-d6c5-42ee-9862-6a2cfeb1605f',...",[],Stadtrat,2022-07-06T21:26:12.1,"[{'Id': 1, 'Name': 'Zuteilung ausstehend'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': '8d6ea382-eeb3-45a8-a7dd-8886...,61720d59-0a1a-45b5-820a-2f0f988f0fdf,https://www.gemeinderat-zuerich.ch/geschaefte/...
1,Zeitlich begrenzte berufliche und schulische A...,2022,108,Postulat,"{'Id': '32e89ef8-f5b2-46e5-aafe-7dd15929f00a',...","{'Id': '6a6e0c52-3ae0-48d3-b7d3-a9412dad103b',...",[],Gemeinderat,None,"[{'Id': 11, 'Name': 'Sozialdepartement (SD)'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': 'bc475206-69c9-4b10-9ebb-55ec...,72a578b2-d377-4b35-9cbf-e72ce850317f,https://www.gemeinderat-zuerich.ch/geschaefte/...
2,Einrichtung einer zeitlich begrenzten Anlaufst...,2022,107,Postulat,"{'Id': '32e89ef8-f5b2-46e5-aafe-7dd15929f00a',...","{'Id': '6a6e0c52-3ae0-48d3-b7d3-a9412dad103b',...",[],Gemeinderat,None,"[{'Id': 11, 'Name': 'Sozialdepartement (SD)'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': '72a578b2-d377-4b35-9cbf-e72c...,bc475206-69c9-4b10-9ebb-55ecddb1c75f,https://www.gemeinderat-zuerich.ch/geschaefte/...


Exportiere den erstellten DataFrame.
Z.K. im DataFrame sieht es aus als ob die URL nicht sauber Concatenated wird. Im Excel-Output ist jedoch alles i.O.

In [24]:
df_deals_details.to_excel('glp_deals_details.xlsx')  
#url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft?gId="df_deals_details['gid']

### Normalisierung der Geschäftsdetails

Der Output der Detailinformationen ist verschachtelt, wie im [Datenmodell](https://opendatazurich.github.io/ris-api/docs/#/Geschaeft/get_Geschaeft_details) beschrieben. In den nächsten Schritten wird der DataFrame mit den Datails zu den Geschäften normalisiert. Siehe dazu z.B folgende Links:
- https://pandas.pydata.org/pandas-docs/version/0.21/generated/pandas.io.json.json_normalize.html
- https://towardsdatascience.com/all-pandas-json-normalize-you-should-know-for-flattening-json-13eae1dfb7dd
- https://blog.softhints.com/python-normalize-json-pandas/


Hier im ersten Versuch werden keine weiteren Parameter angegeben. Es ist jedoch ersichtlich, dass dies nicht dem von uns erhofften Output entspricht. Es ist unklar, was mit den Erstunterzeichnenden geschehen ist ... jedenfalls nützt das so nichts.

In [25]:
from pandas.io.json import json_normalize

In [26]:

#df_deals_details_normalized = pd.json_normalize(deals_details, sep="_")

df_deals_details_normalized = pd.json_normalize(deals_details)
#, max_level=1


df_deals_details_normalized.columns
#df_deals_details_normalized.head(3)
#df_deals_details_normalized.to_excel(path+'deals_details_normalized.xlsx')  

Index(['GeschaeftTitel', 'GeschaeftsJahr', 'GeschaeftsNummer', 'GeschaeftsArt',
       'EinreichendeGruppe', 'PendentBei', 'FristBis', 'Departemente',
       'Ablaufschritte', 'VerknuepfteGeschaefte', 'gid',
       'Erstunterzeichnender.Id', 'Erstunterzeichnender.Name',
       'Erstunterzeichnender.Vorname', 'Erstunterzeichnender.Titel',
       'Erstunterzeichnender.Partei', 'Erstunterzeichnender.Wahlkreis',
       'Erstunterzeichnender.WahlkreisOrderBy',
       'ZweiterErstunterzeichnender.Id', 'ZweiterErstunterzeichnender.Name',
       'ZweiterErstunterzeichnender.Vorname',
       'ZweiterErstunterzeichnender.Titel',
       'ZweiterErstunterzeichnender.Partei',
       'ZweiterErstunterzeichnender.Wahlkreis',
       'ZweiterErstunterzeichnender.WahlkreisOrderBy', 'Erstunterzeichnender',
       'ZweiterErstunterzeichnender'],
      dtype='object')

In [27]:
url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft"
df_deals_details_normalized['deals_details_url'] = url_geschaefte +'?gid='+ df_deals_details_normalized.gid

df_deals_details_normalized.head(2)
df_deals_details_normalized.columns


Index(['GeschaeftTitel', 'GeschaeftsJahr', 'GeschaeftsNummer', 'GeschaeftsArt',
       'EinreichendeGruppe', 'PendentBei', 'FristBis', 'Departemente',
       'Ablaufschritte', 'VerknuepfteGeschaefte', 'gid',
       'Erstunterzeichnender.Id', 'Erstunterzeichnender.Name',
       'Erstunterzeichnender.Vorname', 'Erstunterzeichnender.Titel',
       'Erstunterzeichnender.Partei', 'Erstunterzeichnender.Wahlkreis',
       'Erstunterzeichnender.WahlkreisOrderBy',
       'ZweiterErstunterzeichnender.Id', 'ZweiterErstunterzeichnender.Name',
       'ZweiterErstunterzeichnender.Vorname',
       'ZweiterErstunterzeichnender.Titel',
       'ZweiterErstunterzeichnender.Partei',
       'ZweiterErstunterzeichnender.Wahlkreis',
       'ZweiterErstunterzeichnender.WahlkreisOrderBy', 'Erstunterzeichnender',
       'ZweiterErstunterzeichnender', 'deals_details_url'],
      dtype='object')

In [32]:
df_deals_details_normalized.to_excel('glp_deals_details_normalized.xlsx')  


In [29]:
#deals_details

So können z.B. die Ablaufschritte einzeln noch herausgelesen werden 

In [30]:
df_deals_details_normalized2 = pd.json_normalize(
    deals_details, 
    record_path=['Ablaufschritte'], 
    meta=[['Id', 'Name', 'Vorname']],
    meta_prefix='meta-',
    record_prefix='path-',
    errors='ignore' 
)



In [31]:
#deals_details
df_deals_details_normalized2.head(2)
#df_deals_details_normalized2.columns

,path-Datum,path-Created,path-Titel,path-Protokolleintrag,path-AblaufschrittDokumente,path-BeschlussNrGR,path-SitzungsNr,path-AudioLink,path-AudioLinkText,meta-Id.Name.Vorname
0,2022-04-06T00:00:00,2022-04-06T21:26:44.467,"Eingang, Frist 3 Monate","<p style=""margin: 0px 0px 0px 0px;text-align: ...","[{'Dokumenttitel': '2022_0133.pdf', 'Id': '1f7...",5216.0,193.0,None,None,NaN
1,2022-04-06T00:00:00,2022-04-07T10:48:42.753,"Dringlicherklärung, erfolgt","<p style=""margin: 0px 0px 0px 0px;text-align: ...",[],5191.0,193.0,None,None,NaN
